# 00 Metadata EDA

Audit LAS/LAZ headers, projection zones, duplicates, and QA flags.

In [ ]:
from pathlib import Path
import json
import struct
import numpy as np
import pandas as pd

# Kaggle/local root resolution
if Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

ART = ROOT / 'artifacts'
MANIFEST_DIR = ART / 'manifests'
FEATURE_DIR = ART / 'features'
for p in [ART, MANIFEST_DIR, FEATURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR = ROOT / 'lidar_data'
for candidate in [
    Path('/kaggle/input/lidar-roraima-parime-research/lidar_data'),
    Path('/kaggle/input/lidar-roraima-parime-research'),
]:
    if candidate.exists():
        RAW_DATA_DIR = candidate
        break

print('ROOT:', ROOT)
print('RAW_DATA_DIR:', RAW_DATA_DIR)


In [ ]:
def _clean_ascii(raw: bytes) -> str:
    return raw.split(b"\x00", 1)[0].decode("ascii", errors="ignore").strip()

def _parse_epsg_from_geo_keys(data: bytes):
    if not data or len(data) < 8:
        return None
    vals = struct.unpack('<' + ('H' * (len(data)//2)), data)
    n_keys = vals[3]
    for i in range(n_keys):
        key_id, _, _, value = vals[4 + i*4 : 8 + i*4]
        if key_id == 3072:
            return int(value)
    return None

def parse_laz_header(path: Path):
    with path.open('rb') as f:
        h = f.read(375)
        major, minor = h[24], h[25]
        header_size = struct.unpack_from('<H', h, 94)[0]
        n_vlr = struct.unpack_from('<I', h, 100)[0]
        pdrf_raw = h[104]
        pdrf = int(pdrf_raw & 0x3F)
        compressed = bool(pdrf_raw & 0x80)
        rec_len = struct.unpack_from('<H', h, 105)[0]
        legacy_pc = struct.unpack_from('<I', h, 107)[0]
        point_count = struct.unpack_from('<Q', h, 247)[0] if (major, minor) >= (1, 4) else legacy_pc
        max_x, min_x, max_y, min_y, max_z, min_z = struct.unpack_from('<dddddd', h, 179)

        epsg = None
        f.seek(header_size)
        for _ in range(n_vlr):
            vh = f.read(54)
            if len(vh) < 54:
                break
            user_id = _clean_ascii(vh[2:18])
            rid = struct.unpack_from('<H', vh, 18)[0]
            rl = struct.unpack_from('<H', vh, 20)[0]
            data = f.read(rl)
            if user_id == 'LASF_Projection' and rid == 34735:
                epsg = _parse_epsg_from_geo_keys(data)

    return {
        'tile_id': path.name.replace('.copc.laz', '').replace('.laz', ''),
        'file_name': path.name,
        'file_path': str(path.resolve()),
        'is_copc': path.name.lower().endswith('.copc.laz'),
        'is_duplicate': False,
        'las_version': f'{major}.{minor}',
        'point_format': pdrf,
        'epsg': epsg,
        'point_count': int(point_count),
        'size_bytes': path.stat().st_size,
        'bbox': json.dumps({'min_x':min_x,'max_x':max_x,'min_y':min_y,'max_y':max_y,'min_z':min_z,'max_z':max_z}),
        'qa_flags': json.dumps([]),
    }

paths = sorted({p.resolve() for p in RAW_DATA_DIR.glob('*.laz')})
manifest = pd.DataFrame([parse_laz_header(p) for p in paths]).sort_values(['tile_id','file_name']).reset_index(drop=True)

if not manifest.empty:
    # duplicate by identical bounds+count+epsg
    sig = manifest['point_count'].astype(str) + '|' + manifest['epsg'].astype(str) + '|' + manifest['bbox']
    manifest['is_duplicate'] = sig.duplicated(keep='first')

manifest_path = MANIFEST_DIR / 'tile_manifest.parquet'
manifest.to_parquet(manifest_path, index=False)
print('manifest rows:', len(manifest), 'saved:', manifest_path)
manifest.head()


In [ ]:
manifest["epsg"].value_counts(dropna=False)


In [ ]:
manifest["is_duplicate"].value_counts()


In [ ]:
import json
import matplotlib.pyplot as plt

bbox = manifest["bbox"].apply(json.loads).apply(pd.Series)
plt.figure(figsize=(10, 6))
plt.scatter((bbox["min_x"] + bbox["max_x"]) / 2, (bbox["min_y"] + bbox["max_y"]) / 2, s=30)
plt.title("Tile centroid distribution (projected coordinates)")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()
